## Task 1: ChatGroq Setup 

In [17]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables from .env
load_dotenv()

# Initialize ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

print("ChatGroq initialized successfully!")

ChatGroq initialized successfully!


## Task 2: Basic Chat with ChatGroq

In [18]:
import time

query = "Explain AI in simple words."

start_time = time.perf_counter()

response = llm.invoke(query)

end_time = time.perf_counter()

# Print the response
print("AI Response:")
print(response.content)

# Print latency
latency = end_time - start_time
print(f"\nResponse latency: {latency:.2f} seconds")


AI Response:
**Artificial Intelligence (AI) in simple words**

Imagine you have a very smart robot or computer program that can learn, think, and make decisions—kind of like a brain made out of code. That’s what AI is: a set of computer techniques that let machines do tasks that normally require human intelligence.

Here are the key ideas broken down:

| What humans do | How AI does it |
|----------------|----------------|
| **Seeing** (recognizing faces, objects) | **Computer vision** – looks at pictures and learns what’s in them |
| **Listening** (understanding speech) | **Speech recognition** – turns spoken words into text |
| **Talking** (answering questions) | **Natural language processing** – reads and writes language |
| **Learning** (getting better with practice) | **Machine learning** – finds patterns in data and improves over time |
| **Planning** (choosing the best route, schedule) | **Optimization & decision‑making** – calculates the best option based on goals |

### A quic

## Task 3 : Document Loading & Text Splitting

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_loader = PyPDFLoader("physical.pdf")
docs = pdf_loader.load()

print("Total documents loaded:", len(docs))

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total documents loaded: 10
Total chunks: 34


## Task 4: Embeddings & Vector Store

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Convert chunks into embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Store embeddings and documents in FAISS
vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

print("Documents stored in FAISS:", len(chunks))

# Create a retriever to fetch relevant chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2574.85it/s]


Documents stored in FAISS: 34


## Task 5: RAG Prompt Template

In [22]:
from langchain_core.prompts import ChatPromptTemplate

# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ('system', '''
    You are a helpful assistant.
    Answer the user's question using ONLY the provided context.
    If the answer is not available in the context,
    say: "I don't know based on the provided documents."

    Context:
    {context}
    '''),
    ('human', '{input}')
])

# Test the prompt with sample data
formatted_prompt = prompt.invoke({
    "context": "RAG stands for Retrieval-Augmented Generation.",
    "input": "What does RAG stand for?"
})

print(formatted_prompt)

messages=[SystemMessage(content='\n    You are a helpful assistant.\n    Answer the user\'s question using ONLY the provided context.\n    If the answer is not available in the context,\n    say: "I don\'t know based on the provided documents."\n\n    Context:\n    RAG stands for Retrieval-Augmented Generation.\n    ', additional_kwargs={}, response_metadata={}), HumanMessage(content='What does RAG stand for?', additional_kwargs={}, response_metadata={})]


## Task 6: Build RAG Chain

In [26]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

# Using prev task prompt and llm
# Create the document processing chain
doc_context = create_stuff_documents_chain(llm, prompt)

# Connect retrieval and answer generation
rag_chain = create_retrieval_chain(retriever , doc_context)

# Test with multiple questions
questions = [
    "What is the main topic of the documents?",
    "Explain the most important concept.",
    "What is corona virus?"
]

for question in questions:
    print("\nQuestion:", question)
    response = rag_chain.invoke({'input': question})
    print("Answer:", response['answer'])


Question: What is the main topic of the documents?
Answer: The main topic of the documents is **Health and Physical Education**.

Question: Explain the most important concept.
Answer: The most important concept highlighted in the passage is **physical fitness**.  

Physical fitness is described as the condition in which an individual possesses **great endurance, speed, and strength**. It is presented as the foundation for a **happy, healthy, and vigorous life**. In the context of physical education, developing physical fitness is the primary goal because it enables the body to handle daily demands more efficiently, supports the functioning of internal organ systems, and underpins the other objectives of physical education—social efficiency, sports culture, and mental efficiency. By improving endurance, speed, and strength through regular physical and yogic activities, students become better equipped to cope with physical stresses, maintain overall health, and enjoy an active, fulfilli

## Task 7: Build a Streamlit Chat Interface
Please refer task7.py

## Task 8: Integrate RAG Backend with UI
Please refer task8.py

## Task 9: Multi-Turn Chat Testing

Tested the ChatGroq RAG chatbot with the following scenarios:

Initial factual question: Asked a question related to the uploaded document and verified that the chatbot generated an answer using the retrieved context.

Follow-up question: Asked a follow-up question in the same conversation to test whether the chatbot could understand the previous discussion.

Out-of-context question: Asked a question unrelated to the uploaded document and checked whether the chatbot responded that the information was unavailable in the provided context.

## Task 10 Final ChatGroq RAG App

Developed a Streamlit-based RAG chatbot using ChatGroq, LangChain, Hugging Face embeddings, and FAISS.

Features:

Uploads and processes PDF documents dynamically.

Retrieves relevant document chunks using FAISS.

Generates answers using ChatGroq.

Maintains conversation history using Streamlit session state.

Handles errors during document processing and response generation.

Conclusion: The application provides an interactive interface for document-based question answering.

## Task 11: Observations & Insights

1. Why is Groq suitable for RAG chatbots?

Groq provides fast LLM inference, helping RAG chatbots generate responses with low latency.

2. Difference between Groq RAG and OpenAI RAG

The RAG pipeline is similar, but the LLM provider differs. Groq provides access to supported models through its API, while OpenAI provides its own models and API. Performance, pricing, and available features depend on the selected models and services.

3. Role of Streamlit in GenAI prototyping

Streamlit makes it easy to build interactive web applications with file uploads, chat interfaces, and real-time model responses using Python.

Conclusion: Combining RAG, vector search, an LLM, and Streamlit enables rapid development of document-based AI applications.